# 11 · JAX solid taper — mixed hex+tet segment

The **3-D solid tapered segment** homogenized entirely in **JAX**
(`opensg_jax.fe_jax.solid_taper`): the FEniCS-OpenSG algorithm
(`compute_stiffness(Taper=True)` + `compute_timo_boun`) with **element-type
batches** — hex8 and tet4 volume elements are separate vmapped batches whose COO
triplets concatenate under one global DOF numbering, so a **mixed hex+tet segment
assembles natively into one system**.  The 2-D boundary cross-sections are
**extracted from the 3-D segment** (the JAX analogue of dolfinx `create_submesh`):
hex end-faces give quad4, tet end-faces give tri3 — a mixed segment therefore gets a
**mixed quad+tri boundary**, solved by the same batched 2-D SG (4-mode nullspace KKT).

Case: the tapered tube (thick wall $t/R=0.2$, single-ply $-45^\circ$, $a_R=0.7$,
`taper_study` mesh).  Three solid variants through the SAME solver — all-hex, a
**hybrid** with half the hoop split into tets ($x_2>0$, main-diagonal 6-tet split), and
all-tet — then the **RM shell** ring + tapered segment on the equivalent shell mesh.
Validation of this solver vs FEniCS on identical meshes: **0.008 %** (hex square),
**0.007 %** (hex m45), **0.63 %** (tet ellipse, diagonals ≤ 0.15 %).

In [1]:
import os, sys, time
import numpy as np

ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.insert(0, ROOT); sys.path.insert(0, os.path.join(ROOT, "mitc_rm_segment"))
from opensg_jax.fe_jax.solid_taper import (read_solid_segment_yaml, split_batches_to_tets,
                                           compute_timo_taper_solid_seg)
ORDER = ["EA", "GA2", "GA3", "GJ", "EI2", "EI3"]

def show(title, S, dt=None):
    print("\n%s%s" % (title, ("   [%.2f s]" % dt) if dt is not None else ""))
    for r in range(6):
        print("  " + "  ".join("% .4e" % S[r, c] for c in range(6)))

def pct(S, R):
    cut = np.abs(R).max() / 1e3                      # VABS max/1000 neglect rule
    E = np.full((6, 6), np.nan); m = np.abs(R) > cut
    E[m] = 100.0 * (S[m] - R[m]) / R[m]
    return E

def show_pct(title, E):
    print("\n%s  (%% err, max/1000 rule; . = negligible)" % title)
    for r in range(6):
        print("  " + "  ".join(("%8.3f" % E[r, c]) if np.isfinite(E[r, c]) else "     .  " for c in range(6)))
    print("  max |err| = %.3f %%" % np.nanmax(np.abs(E)))

TAG = "thick_m45_aR070"
MESH = os.path.join(ROOT, "mitc_rm_segment", "out", "taper_study", "meshes")
seg_hex = read_solid_segment_yaml(os.path.join(MESH, "solid_%s.yaml" % TAG))
print("tube %s : %d nodes, %d hex" % (TAG, len(seg_hex["nodes"]), seg_hex["nelem"]))

tube thick_m45_aR070 : 2640 nodes, 1920 hex


## Solid variants — HEX, HYBRID (hex+tet), TET

The hybrid splits every hex with centroid $x_2>0$ into 6 tets (main-diagonal scheme —
face diagonals match between neighbours, so the tet region is conforming; the hex|tet
interface is the standard node-tied transition).  **Both the segment and its extracted
boundaries are mixed-element** for the hybrid.  Each run prints the boundary L/R and
taper segment $6\times6$ with wall times.

In [2]:
conn = seg_hex["batches"]["hex8"][0]
cent_x2 = seg_hex["nodes"][conn].mean(1)[:, 1]
variants = {"HEX": seg_hex,
            "HYBRID": split_batches_to_tets(seg_hex, mask=cent_x2 > 0.0),
            "TET": split_batches_to_tets(seg_hex)}
res = {}
for name, sg in variants.items():
    t0 = time.time()
    DL, DR, DS, info = compute_timo_taper_solid_seg(sg, verbose=False)
    dt = time.time() - t0
    bt = {k: len(v[0]) for k, v in sg["batches"].items()}
    print("\n======== solid %s : batches %s  dof=%d  total %.2f s ========" % (name, bt, info["dof"], dt))
    show("L boundary 6x6:", DL, info["t_boundary"] / 2)
    show("R boundary 6x6:", DR, info["t_boundary"] / 2)
    show("TAPER segment 6x6:", DS, info["t_segment"])
    res[name] = dict(L=DL, R=DR, seg=DS, dt=dt)


======== solid HEX : batches {'hex8': 1920}  dof=7920  total 8.00 s ========

L boundary 6x6:   [1.24 s]
   1.5071e+10   1.4889e-06  -6.3383e-06  -3.8883e+09   2.6010e-06  -8.1773e-07
   1.4889e-06   4.6185e+09  -6.7967e-06  -2.2055e-06   1.9592e+09   8.2395e+06
  -6.3383e-06  -6.6163e-06   4.6185e+09   3.9602e-06  -8.2395e+06   1.9592e+09
  -3.8883e+09  -2.2055e-06   3.9602e-06   9.1149e+09  -2.4384e-06   2.1431e-06
   4.6725e-07   1.9592e+09  -8.2395e+06  -3.0885e-07   7.6061e+09   1.5412e-06
  -5.3329e-06   8.2395e+06   1.9592e+09   3.1997e-06  -6.4804e-07   7.6061e+09

R boundary 6x6:   [1.24 s]
   1.0555e+10   2.0202e-07   3.2823e-07  -1.9223e+09   2.7632e-06   2.9916e-06
   2.0202e-07   3.3122e+09  -3.2364e-05  -2.5073e-07   9.7482e+08   8.3549e+06
   3.2823e-07  -3.2537e-05   3.3122e+09  -9.7501e-07  -8.3549e+06   9.7482e+08
  -1.9223e+09  -2.5073e-07  -9.7501e-07   3.1781e+09  -1.3298e-06  -1.0785e-06
   4.0728e-06   9.7482e+08  -8.3549e+06  -1.9040e-06   2.6409e+09  -8.9186e-


======== solid HYBRID : batches {'hex8': 960, 'tet4': 5760}  dof=7920  total 9.17 s ========

L boundary 6x6:   [1.50 s]
   1.5072e+10   4.3560e+04  -2.0796e+05  -3.8894e+09  -7.9626e+05   4.2651e+05
   4.3560e+04   4.6231e+09   2.0057e+04   2.9016e+05   1.9601e+09   8.7992e+06
  -2.0796e+05   2.0057e+04   4.6231e+09   1.8680e+05  -8.8429e+06   1.9600e+09
  -3.8894e+09   2.9016e+05   1.8680e+05   9.1167e+09   1.1786e+06  -7.5163e+05
  -7.9626e+05   1.9601e+09  -8.8429e+06   1.1786e+06   7.6077e+09  -4.0864e+04
   4.2651e+05   8.7992e+06   1.9600e+09  -7.5163e+05  -4.0864e+04   7.6077e+09

R boundary 6x6:   [1.50 s]
   1.0556e+10  -3.8034e+04  -2.5572e+04  -1.9230e+09  -5.1611e+05   3.1946e+05
  -3.8034e+04   3.3160e+09   1.0701e+04   2.6955e+05   9.7538e+08   8.7648e+06
  -2.5572e+04   1.0701e+04   3.3160e+09  -4.3207e+04  -8.8016e+06   9.7537e+08
  -1.9230e+09   2.6955e+05  -4.3207e+04   3.1789e+09   5.4372e+05  -3.7618e+05
  -5.1611e+05   9.7538e+08  -8.8016e+06   5.4372e+05   2.641


======== solid TET : batches {'tet4': 11520}  dof=7920  total 5.37 s ========

L boundary 6x6:   [0.86 s]
   1.5073e+10   1.0021e-05  -3.2377e-06  -3.8906e+09   2.4450e-05  -9.8694e-06
   1.0021e-05   4.6279e+09   1.9512e-05  -7.6586e-06   1.9610e+09   9.4005e+06
  -3.2377e-06   1.8413e-05   4.6279e+09   1.6050e-06  -9.4005e+06   1.9610e+09
  -3.8906e+09  -7.6586e-06   1.6050e-06   9.1185e+09  -1.7969e-05   6.3624e-06
   2.4427e-05   1.9610e+09  -9.4005e+06  -1.7863e-05   7.6095e+09  -1.0103e-05
  -1.3284e-05   9.4005e+06   1.9610e+09   8.3872e-06   3.2306e-06   7.6095e+09

R boundary 6x6:   [0.86 s]
   1.0557e+10  -4.0913e-06  -4.2695e-06  -1.9238e+09  -2.4839e-06  -5.3168e-06
  -4.0913e-06   3.3199e+09  -8.3533e-06   2.4764e-06   9.7598e+08   9.2164e+06
  -4.2695e-06  -8.3480e-06   3.3199e+09   2.6318e-06  -9.2164e+06   9.7598e+08
  -1.9238e+09   2.4764e-06   2.6318e-06   3.1798e+09   1.9669e-06   3.0502e-06
  -3.9363e-06   9.7598e+08  -9.2164e+06   1.7616e-06   2.6424e+09   3.4467e

## RM shell — boundary rings + tapered shell segment

The equivalent shell mesh (`shell_thick_m45_aR070.yaml`, center reference) through the
general-RM taper machinery (`taper_study.shell_solve`, `mitc4_both`): ring $6\times6$
at L and R plus the tapered shell segment $6\times6$.

In [3]:
t0 = time.time()
from taper_study import shell_solve
CL, S6, CR = shell_solve(TAG)
print("RM shell total %.2f s" % (time.time() - t0))
show("SHELL ring L 6x6:", CL)
show("SHELL ring R 6x6:", CR)
show("SHELL taper segment 6x6:", S6)
res["SHELL"] = dict(L=CL, R=CR, seg=S6)

RM shell total 18.38 s

SHELL ring L 6x6:
   1.5402e+10   8.0003e-06   3.5418e-06  -4.1862e+09   6.2651e-07   5.6753e-06
   8.0003e-06   4.7463e+09   9.4509e-06   1.3831e-06   2.1147e+09  -4.1297e-05
   3.5418e-06   9.4509e-06   4.7218e+09  -1.8057e-05  -1.3051e-05   2.1150e+09
  -4.1862e+09   1.3831e-06  -1.8057e-05   9.3175e+09   1.8947e-05  -8.9533e-07
   6.2651e-07   2.1147e+09  -1.3051e-05   1.8947e-05   7.7186e+09  -9.0018e-06
   5.6753e-06  -4.1297e-05   2.1150e+09  -8.9533e-07  -9.0018e-06   7.7203e+09

SHELL ring R 6x6:
   1.0782e+10  -2.6767e-06  -4.0114e-06  -2.0512e+09  -3.1369e-06   3.5169e-07
  -2.6767e-06   3.3778e+09  -1.1203e-04   5.5164e-05   1.0448e+09  -1.4050e-05
  -4.0114e-06  -1.1203e-04   3.3548e+09  -1.6277e-05   3.5263e-05   1.0443e+09
  -2.0512e+09   5.5164e-05  -1.6277e-05   3.2212e+09   1.6576e-07   5.7840e-07
  -3.1369e-06   1.0448e+09   3.5263e-05   1.6576e-07   2.6588e+09   6.1893e-06
   3.5169e-07  -1.4050e-05   1.0443e+09   5.7840e-07   6.1893e-06   2.

## Comparisons

Hybrid and all-tet vs the all-hex reference (same solver, same geometry), then the RM
shell vs the solid for the boundary rings and the tapered segment.  The thick
($t/R=0.2$) $-45^\circ$ wall is the demanding regime for a shell — diagonals agree to a
few %, the large entries are the $-45^\circ$ coupling terms.

In [4]:
show_pct("solid HYBRID vs solid HEX -- segment", pct(res["HYBRID"]["seg"], res["HEX"]["seg"]))
show_pct("solid TET    vs solid HEX -- segment", pct(res["TET"]["seg"], res["HEX"]["seg"]))
for part, nm in (("L", "L ring"), ("R", "R ring"), ("seg", "TAPER segment")):
    show_pct("RM SHELL vs solid HEX -- %s" % nm, pct(res["SHELL"][part], res["HEX"][part]))


solid HYBRID vs solid HEX -- segment  (% err, max/1000 rule; . = negligible)
     0.017       .         .       0.008       .         .  
       .       0.174       .         .       0.107     0.634
       .         .       0.175       .       0.582     0.110
     0.008       .         .       0.133       .         .  
       .       0.107     0.582       .       0.012       .  
       .       0.634     0.110       .         .       0.013
  max |err| = 0.634 %

solid TET    vs solid HEX -- segment  (% err, max/1000 rule; . = negligible)
     0.033       .         .       0.016       .         .  
       .       0.347       .         .       0.212     1.204
       .         .       0.347       .       1.204     0.212
     0.016       .         .       0.269       .         .  
       .       0.212     1.204       .       0.028       .  
       .       1.204     0.212       .         .       0.028
  max |err| = 1.204 %

RM SHELL vs solid HEX -- L ring  (% err, max/1000 rule; . = negligi